In [3]:
sales_df = spark.read.csv(
    "Files/sales_data.csv",
    header=True,
    inferSchema=True
)

clean_df = sales_df.filter("quantity > 0 AND price > 0")

clean_df.write.mode("overwrite").saveAsTable("sales_clean")

display(clean_df)

StatementMeta(, 97b148bc-595d-40c5-985f-3d766f49b5be, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d0e3faf9-ef91-4617-8a5d-b40a3b714c43)

In [9]:
from pyspark.sql.functions import col, sum as _sum

gold_df = clean_df.groupBy("product_id").agg(
    _sum(col("quantity") * col("price")).alias("total_revenue")
)

gold_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sales_gold")

display(gold_df)

StatementMeta(, 97b148bc-595d-40c5-985f-3d766f49b5be, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9b7aa733-4bdd-4748-8436-3cba091cc34e)

In [8]:
#Streaming Data
import time
from datetime import date

# Get schema from table
schema = spark.table("sales_clean").schema

for i in range(3):
    new_data = spark.createDataFrame(
        [(101, 201, 101, 1, 2, 500, date(2026, 2, 1))],  # ✅ proper DateType
        schema
    )

    new_data.write.mode("append").saveAsTable("sales_clean")

    print("New data ingested...")
    time.sleep(5)

StatementMeta(, 814fe506-273f-4bc8-a7e5-e53d69bf811e, 10, Finished, Available, Finished, False)

New data ingested...
New data ingested...
New data ingested...


In [13]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType
from pyspark.sql.functions import col

# Schema
schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", IntegerType(), True),
    StructField("timestamp", DateType(), True)
])

# READ STREAM
stream_df = spark.readStream \
    .schema(schema) \
    .option("header", True) \
    .csv("Files/streaming_data/")

# DEBUG: Just show incoming data (NO aggregation)
query = stream_df.writeStream \
    .format("memory") \
    .queryName("debug_stream") \
    .outputMode("append") \
    .start()

import time

for i in range(10):
    print(f"\nRefresh {i+1}")
    spark.sql("SELECT * FROM debug_stream").show()
    time.sleep(5)

StatementMeta(, 814fe506-273f-4bc8-a7e5-e53d69bf811e, 15, Finished, Available, Finished, False)


Refresh 1
+--------------+-----------+----------+--------+--------+-----+---------+
|transaction_id|customer_id|product_id|store_id|quantity|price|timestamp|
+--------------+-----------+----------+--------+--------+-----+---------+
+--------------+-----------+----------+--------+--------+-----+---------+


Refresh 2
+--------------+-----------+----------+--------+--------+-----+---------+
|transaction_id|customer_id|product_id|store_id|quantity|price|timestamp|
+--------------+-----------+----------+--------+--------+-----+---------+
+--------------+-----------+----------+--------+--------+-----+---------+


Refresh 3
+--------------+-----------+----------+--------+--------+-----+---------+
|transaction_id|customer_id|product_id|store_id|quantity|price|timestamp|
+--------------+-----------+----------+--------+--------+-----+---------+
+--------------+-----------+----------+--------+--------+-----+---------+


Refresh 4
+--------------+-----------+----------+--------+--------+-----+--

In [14]:
from pyspark.sql.functions import col, sum as _sum

# Add revenue
processed_df = stream_df.withColumn(
    "revenue",
    col("quantity") * col("price")
)

# Aggregate
agg_df = processed_df.groupBy("product_id") \
    .agg(_sum("revenue").alias("total_revenue"))

# Detect spike (LOW threshold for demo)
spike_df = agg_df.filter(
    col("total_revenue") > 1000
)

# Display streaming result
query = spike_df.writeStream \
    .format("memory") \
    .queryName("spike_table") \
    .outputMode("complete") \
    .start()

import time

for i in range(10):
    print(f"\nRefresh {i+1}")
    spark.sql("SELECT * FROM spike_table").show()
    time.sleep(5)

StatementMeta(, 814fe506-273f-4bc8-a7e5-e53d69bf811e, 16, Finished, Available, Finished, False)


Refresh 1
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
+----------+-------------+


Refresh 2
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
+----------+-------------+


Refresh 3
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
+----------+-------------+


Refresh 4
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|       101|        15000|
|       102|         2100|
|       104|         2000|
+----------+-------------+


Refresh 5
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|       101|        15000|
|       102|         2100|
|       104|         2000|
+----------+-------------+


Refresh 6
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|       101|        15000|
|       102|         2100|
|       104|         2000|
+----------+-------------+


Refresh 7
+----------+-------------+

In [17]:
display(spark.table("sales_clean"))

StatementMeta(, 814fe506-273f-4bc8-a7e5-e53d69bf811e, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ef1274fc-7c3d-4fe1-a6f0-71c5b3a5646a)

In [18]:
display(spark.table("sales_gold"))

StatementMeta(, 814fe506-273f-4bc8-a7e5-e53d69bf811e, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bce00587-e97c-4145-8520-449ee91c1214)